# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

### The Target: April `missed_clicks` (Regression)
We are predicting April `missed_clicks`. This gives us a continuous output that inherently serves as the ranking score for our final review queue. To compute this, we use **April's own tier average CTR**, not March's averages. Judging a page against its own month's peers protects the target from platform-wide CTR shifts (like seasonality or algorithm updates) that would unfairly penalize everyone if judged against stale baselines.

### Model Sequence
We will build a simple, readable **Linear Regression** model first. This lets us read the directional weights to verify they make real-world sense before jumping to a complex "black box" like a Random Forest.

### Features and Leakage
We are using 5 March-only signals. These are 100% safe from target leakage. The rule isn't just about "different months"; it's about being **available at the decision moment**. Since an analyst on May 1st has full access to finalized March data, these features are settled facts and do not sneak in any future information about the April outcome.

### Eligible Population
We filter for **>= 500 impressions in both March and April**. In W03/W04 we proved that CTR on < 500 impressions is statistically noisy. Since our April target relies on an April CTR, allowing pages with low April volume would mean evaluating the model against a garbage "answer key."

In [4]:
import duckdb
from dotenv import load_dotenv
import os

# 1. Connect to DuckDB and load the Hugging Face token from your .env file
con = duckdb.connect()
load_dotenv()
HF_TOKEN = os.getenv('HF_TOKEN')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# The path to the daily performance table in the warehouse
FACT_DAILY = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"

# 2. Extract April's data with strict date boundaries and client connectivity filters
april_query = f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions)        AS total_impressions,
        SUM(gsc_clicks)             AS total_clicks,
        AVG(gsc_avg_position)       AS average_position,
        SUM(ga4_sessions)           AS total_sessions,
        SUM(ga4_engaged_sessions)   AS total_engaged_sessions,
        SUM(ga4_pageviews)          AS total_pageviews
    FROM {FACT_DAILY}
    WHERE report_date >= '2026-04-01' AND report_date < '2026-05-01'
      AND client_has_ga4 IS TRUE
      AND client_has_gsc IS TRUE
    GROUP BY content_hash_id, client_hash_id
"""

# 3. Execute the query and load it into a pandas DataFrame
print("Querying April data from Hugging Face... (this may take a moment)")
april_df = con.sql(april_query).df()

# Verify what we pulled
print(f"April Data Shape: {april_df.shape[0]:,} rows, {april_df.shape[1]} columns")
print(april_df.head())


Querying April data from Hugging Face... (this may take a moment)
April Data Shape: 285,820 rows, 8 columns
            content_hash_id           client_hash_id  total_impressions  \
0  content_624f9358ef4f73ba  client_73cda7b4e4f265ea             1015.0   
1  content_953aa51485c651ce  client_73cda7b4e4f265ea             4824.0   
2  content_74f811a044209e62  client_73cda7b4e4f265ea              648.0   
3  content_51bf34b1a55c652f  client_73cda7b4e4f265ea              233.0   
4  content_44dfbe44f41b8ad1  client_73cda7b4e4f265ea              819.0   

   total_clicks  average_position  total_sessions  total_engaged_sessions  \
0           1.0         18.442677             2.0                     0.0   
1          15.0          5.127862            28.0                     0.0   
2           1.0         12.176206             8.0                     0.0   
3           0.0          8.990366             0.0                     0.0   
4           6.0          7.560696            14.0       

In [5]:
# 1. Count unique clients in our April dataframe
april_client_count = april_df['client_hash_id'].nunique()

# 2. Query DuckDB to get the unique client count for March for comparison
march_client_query = f"""
    SELECT COUNT(DISTINCT client_hash_id)
    FROM {FACT_DAILY}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
      AND client_has_ga4 IS TRUE
      AND client_has_gsc IS TRUE
"""
march_client_count = con.sql(march_client_query).fetchone()[0]

print(f"Unique Clients in April: {april_client_count}")
print(f"Unique Clients in March: {march_client_count}")

# 3. Let's go one step further and verify if they are the EXACT same clients
march_clients_query = f"""
    SELECT DISTINCT client_hash_id
    FROM {FACT_DAILY}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
      AND client_has_ga4 IS TRUE
      AND client_has_gsc IS TRUE
"""
march_clients = set([row[0] for row in con.sql(march_clients_query).fetchall()])
april_clients = set(april_df['client_hash_id'].unique())

print(f"Clients dropped (in March but NOT in April): {len(march_clients - april_clients)}")
print(f"New clients (in April but NOT in March): {len(april_clients - march_clients)}")


Unique Clients in April: 48
Unique Clients in March: 43
Clients dropped (in March but NOT in April): 0
New clients (in April but NOT in March): 5


An inner join is used because the model requires March features to make a prediction; the 5 clients that came online in April (with no March history) are correctly excluded, and no existing client's pages were lost due to connectivity changes between the two months


In [8]:
import pandas as pd

# 1. First, extract the March features from the warehouse
march_query = f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions)        AS total_impressions,
        SUM(gsc_clicks)             AS total_clicks,
        AVG(gsc_avg_position)       AS average_position,
        SUM(ga4_sessions)           AS total_sessions,
        SUM(ga4_engaged_sessions)   AS total_engaged_sessions,
        SUM(ga4_pageviews)          AS total_pageviews
    FROM {FACT_DAILY}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
      AND client_has_ga4 IS TRUE
      AND client_has_gsc IS TRUE
    GROUP BY content_hash_id, client_hash_id
"""
print("Querying March data...")
monthly_features = con.sql(march_query).df()

# 2. Now perform the inner join to keep only pages that existed in BOTH months
merged_df = pd.merge(
    monthly_features, 
    april_df, 
    how="inner", 
    on=["client_hash_id", "content_hash_id"], 
    suffixes=('_march', '_april')
)

# Verify the merge
print(f"March rows: {monthly_features.shape[0]:,}")
print(f"April rows: {april_df.shape[0]:,}")
print(f"Merged (Intersection) rows: {merged_df.shape[0]:,}")
print("\nNew columns:")
print(merged_df.columns.tolist())
march_ids = set(monthly_features["content_hash_id"])
merged_ids = set(merged_df["content_hash_id"])
missing = march_ids - merged_ids
print(missing)

Querying March data...
March rows: 260,737
April rows: 285,820
Merged (Intersection) rows: 260,736

New columns:
['content_hash_id', 'client_hash_id', 'total_impressions_march', 'total_clicks_march', 'average_position_march', 'total_sessions_march', 'total_engaged_sessions_march', 'total_pageviews_march', 'total_impressions_april', 'total_clicks_april', 'average_position_april', 'total_sessions_april', 'total_engaged_sessions_april', 'total_pageviews_april']
{'content_0be895ebbcaab26d'}


In [10]:
april_df[april_df['content_hash_id']=='content_0be895ebbcaab26d']

,content_hash_id,client_hash_id,total_impressions,total_clicks,average_position,total_sessions,total_engaged_sessions,total_pageviews


In [11]:
investigate_query = f"""
    SELECT 
        report_date, client_has_ga4, client_has_gsc, gsc_impressions 
    FROM {FACT_DAILY}
    WHERE content_hash_id = 'content_0be895ebbcaab26d' 
      AND report_date >= '2026-04-01' 
      AND report_date < '2026-05-01'
"""
print(con.sql(investigate_query).df())


Empty DataFrame
Columns: [report_date, client_has_ga4, client_has_gsc, gsc_impressions]
Index: []


The inner join drops exactly 1 of 260,737 March pages. Investigation confirmed this page has zero fact-table rows in April entirely (not a connectivity flag issue — the client's other pages report normally), consistent with the page being deleted or unpublished between March and April. This is expected content churn, not a data quality issue

In [12]:
import numpy as np

# 1. Apply the strict eligibility filter (>= 500 impressions in BOTH months)
eligible_df = merged_df[
    (merged_df['total_impressions_march'] >= 500) & 
    (merged_df['total_impressions_april'] >= 500)
].copy()

print(f"Eligible pages for modeling: {eligible_df.shape[0]:,}\n")

# 2. Calculate April's actual CTR (with the zero-impression NaN guard)
eligible_df['ctr_april'] = np.where(
    eligible_df['total_impressions_april'] > 0, 
    eligible_df['total_clicks_april'] / eligible_df['total_impressions_april'], 
    np.nan
)

# 3. Create April's position_tier (using the exact W04 bins/labels)
bins = [-float("inf"), 3, 10, 20, 50, float("inf")]
labels = ["top_3", "page_1", "striking", "page_3_5", "deep"]

eligible_df["position_tier_april"] = pd.cut(eligible_df["average_position_april"], bins=bins, labels=labels)
eligible_df["position_tier_april"] = eligible_df["position_tier_april"].cat.add_categories(["no_data"]).fillna("no_data")

# 4. Calculate April's tier_avg_ctr (grouping by April's own tiers)
raw_ctr_by_pos_april = eligible_df.groupby("position_tier_april", observed=False)["ctr_april"].mean()
eligible_df["tier_avg_ctr_april"] = eligible_df["position_tier_april"].map(raw_ctr_by_pos_april)

# Let's peek at the final calculated columns
print("April Math Verification:")
print(eligible_df[["content_hash_id", "total_impressions_april", "ctr_april", "position_tier_april", "tier_avg_ctr_april"]].head())


Eligible pages for modeling: 37,004

April Math Verification:
              content_hash_id  total_impressions_april  ctr_april  \
20   content_ad0905e0e6b116bc                   2184.0   0.002289   
21   content_25f45960c8e060cd                    616.0   0.000000   
116  content_7b50820e76b1a006                   1736.0   0.012673   
226  content_6ca4da7864d59bd4                   1588.0   0.004408   
309  content_15770c63daac443b                  10112.0   0.001187   

    position_tier_april  tier_avg_ctr_april  
20               page_1            0.003254  
21             page_3_5            0.001271  
116                deep            0.000432  
226              page_1            0.003254  
309            striking            0.002840  


In [13]:
eligible_df["position_tier_april"].value_counts()

position_tier_april
page_1      16639
page_3_5    10267
striking     8916
top_3         952
deep          230
no_data         0
Name: count, dtype: int64

In [15]:
# 1. Calculate the gap (How much worse did this page perform vs. its tier average?)
eligible_df['ctr_gap_april'] = eligible_df['tier_avg_ctr_april'] - eligible_df['ctr_april']

# 2. Clip at 0 (If a page BEAT its tier average, the gap is negative. We turn negative numbers into 0 so we don't accidentally calculate "negative missed clicks").
eligible_df['ctr_gap_april'] = eligible_df['ctr_gap_april'].clip(lower=0)

# 3. Multiply the gap by actual impressions to get the final raw number of missed clicks
eligible_df['missed_clicks_april'] = eligible_df['ctr_gap_april'] * eligible_df['total_impressions_april']

# Verification
print("Target created! Here are the top 5 pages by actual April missed clicks:")
print(eligible_df[['content_hash_id', 'tier_avg_ctr_april', 'ctr_april', 'ctr_gap_april', 'missed_clicks_april']].sort_values('missed_clicks_april', ascending=False).head())
eligible_df["missed_clicks_april"].describe()

Target created! Here are the top 5 pages by actual April missed clicks:
                 content_hash_id  tier_avg_ctr_april  ctr_april  \
132615  content_0e03de7680314cd5            0.005015   0.002397   
141913  content_99fc6465edb0e52c            0.002840   0.000004   
240750  content_39e19a3ec2d95f9d            0.003254   0.000011   
141666  content_62770e1299963fe4            0.003254   0.000999   
99985   content_77276ad7a26f4905            0.003254   0.001255   

        ctr_gap_april  missed_clicks_april  
132615       0.002617           799.174869  
141913       0.002837           760.126319  
240750       0.003243           606.060065  
141666       0.002255           514.370691  
99985        0.001999           454.084266  


count    37004.000000
mean         5.239647
std         16.107787
min          0.000000
25%          0.000000
50%          1.393662
75%          4.590166
max        799.174869
Name: missed_clicks_april, dtype: float64

In [18]:
# Create the ML-ready target using log1p to handle the heavy-tailed whales safely
eligible_df['target_log_missed_clicks'] = np.log1p(eligible_df['missed_clicks_april'])

# Log transform all the heavy-tailed March volume features (our X inputs)
eligible_df['log_impressions_march'] = np.log1p(eligible_df['total_impressions_march'])
eligible_df['log_clicks_march'] = np.log1p(eligible_df['total_clicks_march'])
eligible_df['log_sessions_march'] = np.log1p(eligible_df['total_sessions_march'])
eligible_df['log_engaged_sessions_march'] = np.log1p(eligible_df['total_engaged_sessions_march'])
eligible_df['log_pageviews_march'] = np.log1p(eligible_df['total_pageviews_march'])

print("Section 1 Complete! Here is a peek at the logged Target and Inputs:")
print(eligible_df[['missed_clicks_april', 'target_log_missed_clicks', 'total_impressions_march', 'log_impressions_march']].head())
eligible_df["target_log_missed_clicks"].describe()


Section 1 Complete! Here is a peek at the logged Target and Inputs:
     missed_clicks_april  target_log_missed_clicks  total_impressions_march  \
20              2.106486                  1.133492                   1844.0   
21              0.782949                  0.578269                   3002.0   
116             0.000000                  0.000000                   2243.0   
226             0.000000                  0.000000                    867.0   
309            16.722605                  2.874841                   2585.0   

     log_impressions_march  
20                7.520235  
21                8.007367  
116               7.716015  
226               6.766192  
309               7.857868  


count    37004.000000
mean         1.043175
std          1.080240
min          0.000000
25%          0.000000
50%          0.872825
75%          1.721009
max          6.684830
Name: target_log_missed_clicks, dtype: float64

## 2. Split design

### Time-Aware Split (The Decision Point)
Our split simulates a real-world decision: using **March** data (features) to predict an **April** outcome (target). This guarantees no time-travel leakage.

### Grouped by Client
We group the train/test split by `client_hash_id`. As we discovered in W03, a single dominant client accounts for roughly 31,887 pages. A random split would bleed this client into both training and testing, allowing the model to cheat by simply memorizing that client's site structure rather than learning generalizable SEO rules.


*(Note: This is a single grouped train/test split. Results may vary depending on which clients happen to land in the test set. A full GroupKFold cross-validation would be the rigorous next step to ensure stability across clients).*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

### The Evaluation Contract (Graded against April)
Both the baseline rule and the ML model make their predictions using ONLY March data. However, both will be evaluated against the **exact same April ground truth**. (Evaluating the March baseline against March data would be circular and trivially yield a perfect score).

We use two metrics side-by-side on the Top 50 recommended pages:
1. **Precision@50 (Binary):** What percentage of the Top 50 *actually* met the baseline rule's condition in April? (Actual April CTR < 50% of April Tier Average, with >= 500 April impressions). This reuses the exact same threshold from W04 to ensure a fair, apples-to-apples comparison.
2. **Total Captured Missed Clicks @ 50 (Business Value):** What is the sum of actual April `missed_clicks` captured by the Top 50 recommendations? This tells the business the real value of the prioritized list.


### Base Rate & Tie Policy
- **Base Rate:** Before evaluating Precision@50, we must compute the background base rate (what fraction of ALL eligible pages actually met the April opportunity condition?). This provides a baseline for random guessing, letting us know if our Precision@50 is actually impressive.
- **Tie Policy:** If the model outputs identical predicted `missed_clicks` scores for multiple pages right at the Top 50 cutoff, we will break ties by sorting alphabetically by `content_hash_id`. This prevents Pandas from making silent ranking decisions based on original row order.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

### Planned Sensitivity Check: The Volume Dominance
Our target (`missed_clicks`) multiplies a CTR gap by `total_impressions`. The risk is that the model takes the lazy route: *"big pages stay big."* March `total_impressions` and `total_clicks` are perfectly legal features, but if the model leans entirely on them, its "insight" is just restating scale, not finding SEO opportunity.

**The Test:** We must train the model *with* vs. *without* March volume metrics (`total_impressions` and `total_clicks`). We need to see how much the model's advantage shrinks when we remove these "close cousin" scale features, proving whether it is actually learning nuanced SEO patterns or just acting as a simple traffic forecaster.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.